# Model Summary
Сводка по архитектурам **SimpleCNNClassifier** и **UNetNoPadding** (v2) и их legacy-аналогам **SmallResNet** / **SimpleUNet** (v1).

In [14]:
import sys, os
# Добавляем корень проекта в путь — запускайте ноутбук из корня AIP
sys.path.insert(0, os.path.abspath('..'))

import torch
import pandas as pd
from IPython.display import display

# torchinfo — pip install torchinfo (если ещё не установлен)
try:
    from torchinfo import summary as ti_summary
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchinfo', '-q'])
    from torchinfo import summary as ti_summary

# ── импорт моделей ──────────────────────────────────────────────────
from src.models.classification import SimpleCNNClassifier
from src.models.segmentation   import UNetNoPadding
from src.models.legacy_aip_v1  import SmallResNet, SimpleUNet

In [15]:
def count_params(model: torch.nn.Module):
    """Возвращает (total, trainable) параметров."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def layer_table(model: torch.nn.Module) -> pd.DataFrame:
    """Детальная таблица: имя слоя, тип, параметры."""
    rows = []
    for name, module in model.named_modules():
        if not list(module.children()):          # только листья
            params = sum(p.numel() for p in module.parameters())
            rows.append({
                'Layer': name or '(root)',
                'Type' : module.__class__.__name__,
                'Params': params,
            })
    return pd.DataFrame(rows)


def summary_table(models_dict: dict) -> pd.DataFrame:
    """Сводная таблица по всем моделям."""
    rows = []
    for label, model in models_dict.items():
        total, trainable = count_params(model)
        rows.append({
            'Model'            : label,
            'Total params'     : f'{total:,}',
            'Trainable params' : f'{trainable:,}',
            'Non-trainable'    : f'{total - trainable:,}',
        })
    return pd.DataFrame(rows)

In [16]:
# ── Инициализация моделей ───────────────────────────────────────────
models = {
    'SimpleCNNClassifier (v2, BN)' : SimpleCNNClassifier(num_classes=4, norm_type='bn'),
    'SimpleCNNClassifier (v2, GN)' : SimpleCNNClassifier(num_classes=4, norm_type='gn'),
    'SmallResNet (v1)'             : SmallResNet(num_classes=4),
    'UNetNoPadding (v2, GN)'       : UNetNoPadding(in_ch=1, base=32, out_ch=1, norm_type='gn'),
    'UNetNoPadding (v2, BN)'       : UNetNoPadding(in_ch=1, base=32, out_ch=1, norm_type='bn'),
    'SimpleUNet (v1)'              : SimpleUNet(in_ch=1, base=32, out_ch=1),
}

## Сводная таблица

In [17]:
df_summary = summary_table(models)
display(df_summary.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1C4E80'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('text-align', 'left')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f2f2')]},
]).hide(axis='index'))

Model,Total params,Trainable params,Non-trainable
"SimpleCNNClassifier (v2, BN)","886,244","886,244",0
"SimpleCNNClassifier (v2, GN)","886,244","886,244",0
SmallResNet (v1),"2,796,772","2,796,772",0
"UNetNoPadding (v2, GN)","1,926,433","1,926,433",0
"UNetNoPadding (v2, BN)","1,926,433","1,926,433",0
SimpleUNet (v1),"1,926,433","1,926,433",0


## Детальная разбивка по слоям

Разверните нужную модель ниже.

In [18]:
for name, model in models.items():
    total, _ = count_params(model)
    print(f'\n{'='*60}')
    print(f'  {name}   ({total:,} params total)')
    print('='*60)
    df = layer_table(model)
    display(df.style
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1C4E80'), ('color', 'white')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f2f2')]},
        ])
        .hide(axis='index')
        .format({'Params': '{:,}'})
        .bar(subset=['Params'], color='#90CAF9')
    )


  SimpleCNNClassifier (v2, BN)   (886,244 params total)


Layer,Type,Params
stem.0,Conv2d,288
stem.1,BatchNorm2d,64
stem.2,GELU,0
stage1.conv1,Conv2d,"9,216"
stage1.bn1,BatchNorm2d,64
stage1.act,GELU,0
stage1.conv2,Conv2d,"9,216"
stage1.bn2,BatchNorm2d,64
stage1.shortcut,Identity,0
stage2.conv1,Conv2d,"18,432"



  SimpleCNNClassifier (v2, GN)   (886,244 params total)


Layer,Type,Params
stem.0,Conv2d,288
stem.1,GroupNorm,64
stem.2,GELU,0
stage1.conv1,Conv2d,"9,216"
stage1.bn1,GroupNorm,64
stage1.act,GELU,0
stage1.conv2,Conv2d,"9,216"
stage1.bn2,GroupNorm,64
stage1.shortcut,Identity,0
stage2.conv1,Conv2d,"18,432"



  SmallResNet (v1)   (2,796,772 params total)


Layer,Type,Params
stem.0,Conv2d,"1,568"
stem.1,BatchNorm2d,64
stem.2,ReLU,0
stem.3,MaxPool2d,0
layer1.0.conv1,Conv2d,"9,216"
layer1.0.bn1,BatchNorm2d,64
layer1.0.conv2,Conv2d,"9,216"
layer1.0.bn2,BatchNorm2d,64
layer1.0.shortcut,Identity,0
layer1.1.conv1,Conv2d,"9,216"


... (2 outputs left out)


  UNetNoPadding (v2, BN)   (1,926,433 params total)


Layer,Type,Params
down1.net.0,Conv2d,288
down1.net.1,BatchNorm2d,64
down1.net.2,GELU,0
down1.net.3,Conv2d,"9,216"
down1.net.4,BatchNorm2d,64
down1.net.5,GELU,0
down1.net.6,Identity,0
pool1,MaxPool2d,0
down2.net.0,Conv2d,"18,432"
down2.net.1,BatchNorm2d,128



  SimpleUNet (v1)   (1,926,433 params total)


Layer,Type,Params
down1.net.0,Conv2d,288
down1.net.1,BatchNorm2d,64
down1.net.2,ReLU,0
down1.net.3,Conv2d,"9,216"
down1.net.4,BatchNorm2d,64
down1.net.5,ReLU,0
pool1,MaxPool2d,0
down2.net.0,Conv2d,"18,432"
down2.net.1,BatchNorm2d,128
down2.net.2,ReLU,0


In [19]:
# Входные размеры: классификаторы — (B, 1, 224, 224), сегментация — (B, 1, 256, 256)
INPUT_SHAPES = {
    'SimpleCNNClassifier (v2, BN)' : (1, 1, 224, 224),
    'SimpleCNNClassifier (v2, GN)' : (1, 1, 224, 224),
    'SmallResNet (v1)'             : (1, 1, 224, 224),
    'UNetNoPadding (v2, GN)'       : (1, 1, 256, 256),
    'UNetNoPadding (v2, BN)'       : (1, 1, 256, 256),
    'SimpleUNet (v1)'              : (1, 1, 256, 256),
}

for name, model in models.items():
    print(f'\n{"="*70}')
    print(f'  {name}')
    print('='*70)
    ti_summary(
        model,
        input_size=INPUT_SHAPES[name],
        col_names=['input_size', 'output_size', 'num_params', 'mult_adds'],
        row_settings=['var_names'],
        depth=4,
        verbose=1,
    )


  SimpleCNNClassifier (v2, BN)
Layer (type (var_name))                       Input Shape               Output Shape              Param #                   Mult-Adds
SimpleCNNClassifier (SimpleCNNClassifier)     [1, 1, 224, 224]          [1, 4]                    --                        --
├─Sequential (stem)                           [1, 1, 224, 224]          [1, 32, 112, 112]         --                        --
│    └─Conv2d (0)                             [1, 1, 224, 224]          [1, 32, 112, 112]         288                       3,612,672
│    └─BatchNorm2d (1)                        [1, 32, 112, 112]         [1, 32, 112, 112]         64                        64
│    └─GELU (2)                               [1, 32, 112, 112]         [1, 32, 112, 112]         --                        --
├─ResidualConvBlock (stage1)                  [1, 32, 112, 112]         [1, 32, 112, 112]         --                        --
│    └─Identity (shortcut)                    [1, 32, 112, 112]  